Perform Text Generation

In this notebook, you learn how to use **Large Language Model (LLM)** to generate an email response to a customer who provided negative feedback on the quality of customer service they received from the support engineer. In this notebook, you generate an email with a thank you note based on the customer's previous email. You use the **Amazon Titan model** using the **Amazon Bedrock API** with **Boto3 client**.

The prompt used in this task is called a **zero-shot prompt**. In a zero-shot prompt, you describe the task or desired output to the language model in plain language. The model then uses its pre-trained knowledge and capabilities to generate a response or complete the task based solely on the provided prompt. In short, the only content received by the LLM is the user message without any external other context.

## Task 1: Environment setup

In this task, you set up your environment.

In [1]:
import json
import os
import sys

module_path = ".."
sys.path.append(os.path.abspath(module_path))

The `boto3.client('bedrock-runtime')` method creates a client for interacting with Amazon Bedrock's model inference APIs. It accepts the following:

- `service_name`: The AWS service to connect to, in this case, `bedrock-runtime`.
- `region_name`: The AWS Region where the service is available. It defaults to the value of the `AWS_DEFAULT_REGION` environment variable, or `None` if the variable is not set, allowing Boto3 to resolve the region from its default configuration.

Refer to the [Boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/core/session.html?utm_source=chatgpt.com#boto3.session.Session.client) for more information.

> Note: **boto3** is the AWS SDK for Python, providing high-level interfaces to interact with AWS services such as Amazon Bedrock, S3, and SQS. While, botocore is the low-level core library **used by Boto3**, responsible for AWS API requests, request signing, configuration, credential handling, and service responses. It also provides the underlying exception classes used to handle AWS errors. **You can import botocore when you want to handle specifics exceptions as`botocore.exceptions.ClientError`**.

In [2]:
import boto3

bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name=os.environ.get("AWS_DEFAULT_REGION"),
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY")
)

Amazon Bedrock provides four main methods for model inference: `converse`, `converse_stream`, `invoke_model`, and `invoke_model_with_response_stream`. Refer to the [Nova API examples](https://docs.aws.amazon.com/nova/latest/nova2-userguide/code-library.html#converse-api-examples) for method usage.

The `invoke_model` and `invoke_model_with_response_stream` methods accept the following parameters:
- `body`: A JSON string containing the input prompt and model-specific inference parameters.
- `modelId`: The model ID or ARN identifying the foundation model or inference resource to invoke.
- `accept`: The desired response format, specified as a MIME type.
- `contentType`: The MIME type of the request body.

In this notebook, we will use `converse` and `invoke_model_with_response_stream`. The main difference between the `converse*` and `invoke_model*` methods is the level of abstraction and the request/response format: `converse*` methods provide a standardized interface across supported models, whereas `invoke_model*` methods require model-specific request and response formats.

The streaming variants of both methods differ from their non-streaming counterparts in how they deliver the response: they emit incremental events as the model generates output rather than returning the complete response at once.

Additional resources:
- [Available foundation model IDs and ARNs](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids-arns.html)
- [Amazon Nova Lite model reference](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-lite.html)

In [5]:
modelId = 'amazon.nova-micro-v1:0'
accept = 'application/json'
contentType = 'application/json'
inferenceConfig = {
    "maxTokens": 8192,
    "temperature": 0,
    "topP": 0.9,
    "stopSequences": [],
}

## Task 2: Generate text

In this task, we'll prepare an input for the Amazon Bedrock service to generate an email. We'll explore how the model generates an output based on the prompt created earlier and how the model generates an output based on the prompt created earlier.

This email is generated using the Amazon Nova model by understanding the input request and utilizing its inherent understanding of different modalities. The request to the API is synchronous and waits for the entire output to be generated by the model.

> **Note:** This notebook does not handle exceptions explicitly; they are allowed to propagate naturally. To handle specific exceptions, refer to the [Botocore exception reference](https://botocore.amazonaws.com/v1/documentation/api/latest/reference/exceptions.html).

In [6]:
prompt_data = """
Write an email from Bob, Customer Service Manager, AnyCompany to the customer "John Doe" 
who provided negative feedback on the service provided by our customer support 
engineer
"""

response = bedrock_client.converse(
    modelId=modelId,
    inferenceConfig=inferenceConfig,
    messages=[
        {
            "role": "user",
            "content": [{"text": prompt_data}],
        }
    ],
)

outputText = response["output"]["message"]["content"][0]["text"]

print(outputText)

Subject: Addressing Your Feedback and Ensuring Improved Service

Dear John Doe,

I hope this email finds you well. I am writing to you on behalf of AnyCompany's Customer Service team to address the feedback you recently provided regarding the service you received from our customer support engineer.

First and foremost, I would like to extend my sincerest apologies for any inconvenience or dissatisfaction you experienced. At AnyCompany, we strive to provide exceptional service to all our customers, and it is clear that we fell short in meeting your expectations on this occasion.

We take all feedback very seriously, and your comments have been carefully reviewed by our team. Here are some key points we have taken from your feedback:

[Insert specific points from John's feedback here, if known]

To ensure that this does not happen again, we are taking the following steps:

1. **Training and Development:** We are implementing additional training for our customer support engineers to bette

### Streaming Output Generation

Bedrock also supports that the output can be streamed as it is generated by the model in form of chunks. This email is generated by invoking the model with streaming option. `invoke_model_with_response_stream` returns a `ResponseStream` which you can read from.

**Reference: https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-lite.html**

> Note: The response format of `invoke_model` and `invoke_model_with_response_stream` varies depending on the model being invoked. For a more consistent response format across supported models, prefer using `converse` or `converse_stream`.

In [7]:
user_message = 'Can you explain the features of Amazon Bedrock?'
print("*User:*",  user_message, end="\n\n")

body_message = json.dumps({
            'messages': [{
                'role': 'user',
                'content': [{'text': user_message}]
            }],
            'inferenceConfig': inferenceConfig
    })

response = bedrock_client.invoke_model_with_response_stream(
    body=body_message, 
    modelId=modelId,
    accept=accept, 
    contentType=contentType
)
stream = response.get('body')

print("*Assistant:*", end=" ")

if stream:
    for event in stream:
        chunk_obj = event.get('chunk').get('bytes')

        if delta_event := json.loads(chunk_obj).get('contentBlockDelta'):
            text = delta_event.get("delta").get("text")
            print(text, end="")


*User:* Can you explain the features of Amazon Bedrock?

*Assistant:* Amazon Bedrock is a foundational service designed to help developers build, test, and deploy machine learning models at scale. Here are some of its key features:

### 1. **Scalability**
   - **Elastic Compute Resources**: Amazon Bedrock leverages AWS's extensive infrastructure to provide scalable compute resources, allowing you to handle large datasets and complex models efficiently.
   - **Auto-scaling**: It automatically adjusts resources based on the workload, ensuring optimal performance and cost-efficiency.

### 2. **Integration with AWS Ecosystem**
   - **AWS Services**: Seamlessly integrates with other AWS services like Amazon S3 for data storage, Amazon SageMaker for model training, and Amazon Redshift for data warehousing.
   - **APIs and SDKs**: Provides robust APIs and Software Development Kits (SDKs) for easy integration into existing workflows.

### 3. **Managed Services**
   - **Fully Managed**: Reduces

The stream with response approach helps to quickly obtain the output of the model and allows the service to complete it as you read. This assists in use cases where you request the model to generate longer pieces of text. You can later combine all the chunks generated to form the complete output and use it for your use case. 

You have now experimented with using the boto3 SDK, which provides basic exposure to the Amazon Bedrock API. Using this API, you have seen the use case of generating an email to respond to a customer's negative feedback.

### Try it yourself
- Change the prompts to your specific usecase and evaluate the output of different models.
- Play with the token length to understand the latency and responsiveness of the service.
- Apply different prompt engineering principles to get better outputs.

I hope you enjoyed it!

Visit my [GitHub profile](https://github.com/rafaself).